In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import holidays
import hashlib
import tools


In [2]:
df_kalendarz = pd.read_parquet("dane/interim/fact_inka_hard_flagged_wazone.parquet")


In [3]:
# ============================================================
# Pełny kalendarz kalendarzowy z flagą otwarcia sklepu
# ============================================================
df_sprzedaz = df_kalendarz[df_kalendarz['TypRuchu'] == 'sprzedaz'].copy()

# 1. Zakres dat z danych
data_min = df_kalendarz['Data'].min()
data_max = df_kalendarz['Data'].max()

kalendarz_dni_pelny = pd.date_range(start=data_min, end=data_max, freq='D')
print(f"Wszystkich dni w zakresie: {len(kalendarz_dni_pelny)}")

# 2. Dni otwarcia sklepu - wywnioskowane z rzeczywistych transakcji sprzedaży
dni_otwarte = set(df_sprzedaz['Data'].dt.normalize().unique())
print(f"Dni z realną sprzedażą (otwarte): {len(dni_otwarte)}")

# 3. Budujemy DataFrame kalendarza dni z podstawowymi atrybutami
df_dni_kalendarz = pd.DataFrame({'Data': kalendarz_dni_pelny})

df_dni_kalendarz['CzySklepOtwarty'] = df_dni_kalendarz['Data'].isin(dni_otwarte)
df_dni_kalendarz['DzienTygodnia'] = df_dni_kalendarz['Data'].dt.dayofweek  # 0=pon, 6=niedz
df_dni_kalendarz['NazwaDnia'] = df_dni_kalendarz['Data'].dt.day_name()
df_dni_kalendarz['CzyNiedziela'] = df_dni_kalendarz['DzienTygodnia'] == 6
df_dni_kalendarz['CzySobota'] = df_dni_kalendarz['DzienTygodnia'] == 5

print(f"\nRozkład otwarcia wg dnia tygodnia:")
print(df_dni_kalendarz.groupby('NazwaDnia')['CzySklepOtwarty'].agg(['sum', 'count']))


Wszystkich dni w zakresie: 1127
Dni z realną sprzedażą (otwarte): 952

Rozkład otwarcia wg dnia tygodnia:
           sum  count
NazwaDnia            
Friday     156    161
Monday     153    161
Saturday   157    161
Sunday      21    161
Thursday   153    161
Tuesday    157    161
Wednesday  155    161


In [4]:
pl_holidays = holidays.Poland(years=range(data_min.year, data_max.year + 1))

df_dni_kalendarz['CzySwieto'] = df_dni_kalendarz['Data'].apply(lambda d: d in pl_holidays)
df_dni_kalendarz['NazwaSwieta'] = df_dni_kalendarz['Data'].apply(lambda d: pl_holidays.get(d, None))

print(f"\nDni świąteczne w zakresie: {df_dni_kalendarz['CzySwieto'].sum()}")
print(df_dni_kalendarz[df_dni_kalendarz['CzySwieto']][['Data', 'NazwaSwieta', 'CzySklepOtwarty']].head(20))



Dni świąteczne w zakresie: 42
          Data                       NazwaSwieta  CzySklepOtwarty
0   2023-01-01                    New Year's Day            False
5   2023-01-06                          Epiphany            False
98  2023-04-09                     Easter Sunday            False
99  2023-04-10                     Easter Monday            False
120 2023-05-01                      National Day            False
122 2023-05-03  National Day of the Third of May            False
147 2023-05-28                         Pentecost            False
158 2023-06-08                    Corpus Christi            False
226 2023-08-15                    Assumption Day            False
304 2023-11-01                   All Saints' Day            False
314 2023-11-11         National Independence Day            False
358 2023-12-25                     Christmas Day            False
359 2023-12-26           Second Day of Christmas            False
365 2024-01-01                    New Year's 

In [5]:
# ============================================================
# Budowa pełnego kalendarza TowId × Data — Z PRZYCIĘCIEM do okresu życia
#
# ============================================================

towid_do_kalendarza = df_sprzedaz['TowId'].unique()

print(f"TowId do kalendarza (z jakąkolwiek sprzedażą): {len(towid_do_kalendarza)}")

# 1. Pełny cross join
pelny_kalendarz = pd.MultiIndex.from_product(
    [towid_do_kalendarza, kalendarz_dni_pelny],
    names=['TowId', 'Data']
).to_frame(index=False)

print(f"Rozmiar przed przycięciem: {len(pelny_kalendarz):,} wierszy")

# 2. PRZYCINAMY od razu — okres życia per TowId (+bufor)
zakres_zycia = df_sprzedaz.groupby('TowId')['Data'].agg(['min', 'max']).reset_index()
zakres_zycia.columns = ['TowId', 'PierwszaSprzedaz', 'OstatniaSprzedazTowId']

pelny_kalendarz = pelny_kalendarz.merge(zakres_zycia, on='TowId', how='left')

bufor_dni = 30
pelny_kalendarz = pelny_kalendarz[
    (pelny_kalendarz['Data'] >= pelny_kalendarz['PierwszaSprzedaz'] - pd.Timedelta(days=bufor_dni)) &
    (pelny_kalendarz['Data'] <= pelny_kalendarz['OstatniaSprzedazTowId'] + pd.Timedelta(days=bufor_dni))
].copy()

print(f"Rozmiar po przycięciu:     {len(pelny_kalendarz):,} wierszy")

# 3. Reszta merge'y — na mniejszym, przyciętym zbiorze
pelny_kalendarz = pelny_kalendarz.merge(df_dni_kalendarz, on='Data', how='left')

sprzedaz_dzienna = df_sprzedaz.groupby(['TowId', 'Data']).agg(
    Wartosc=('Wartosc', 'sum'),
    Ilosc=('IloscPlus', 'sum'),
    LiczbaTransakcji=('TowId', 'count')
).reset_index()

pelny_kalendarz = pelny_kalendarz.merge(sprzedaz_dzienna, on=['TowId', 'Data'], how='left')
pelny_kalendarz['Wartosc'] = pelny_kalendarz['Wartosc'].fillna(0)
pelny_kalendarz['Ilosc'] = pelny_kalendarz['Ilosc'].fillna(0)
pelny_kalendarz['LiczbaTransakcji'] = pelny_kalendarz['LiczbaTransakcji'].fillna(0).astype(int)

# 4. Cena dzienna + forward-fill
cena_dzienna = df_sprzedaz.groupby(['TowId', 'Data']).agg(
    SumaWartosci=('Wartosc', 'sum'),
    SumaIlosci=('IloscPlus', 'sum')
).reset_index()
cena_dzienna['CenaDzienna'] = cena_dzienna['SumaWartosci'] / cena_dzienna['SumaIlosci']
cena_dzienna = cena_dzienna[['TowId', 'Data', 'CenaDzienna']]

pelny_kalendarz = pelny_kalendarz.merge(cena_dzienna, on=['TowId', 'Data'], how='left')
pelny_kalendarz = pelny_kalendarz.sort_values(['TowId', 'Data'])
pelny_kalendarz['CenaOstatniaZnana'] = pelny_kalendarz.groupby('TowId')['CenaDzienna'].ffill()
pelny_kalendarz['ProduktJeszczeNieSprzedawany'] = pelny_kalendarz['CenaOstatniaZnana'].isna()

# 5. Nazwy i flaga ważenia (z df_kalendarz, bez KategoriaRotacji)
pelny_kalendarz = pelny_kalendarz.merge(
    df_kalendarz[['TowId', 'NazwaTow']].drop_duplicates(subset='TowId'),
    on='TowId', how='left'
)
pelny_kalendarz = pelny_kalendarz.merge(
    df_kalendarz[['TowId', 'JestWazony']].drop_duplicates(subset='TowId'),
    on='TowId', how='left'
)


TowId do kalendarza (z jakąkolwiek sprzedażą): 12481
Rozmiar przed przycięciem: 14,066,087 wierszy
Rozmiar po przycięciu:     7,429,302 wierszy


In [6]:
print(f"Rozmiar: {pelny_kalendarz.shape}")
print(f"Pamięć: {pelny_kalendarz.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nKolumny: {pelny_kalendarz.columns.tolist()}")

# Szybki sanity-check przed zapisem
print(f"\nBrakujące wartości per kolumna:")
print(pelny_kalendarz.isna().sum())


Rozmiar: (7429302, 19)
Pamięć: 1063.3 MB

Kolumny: ['TowId', 'Data', 'PierwszaSprzedaz', 'OstatniaSprzedazTowId', 'CzySklepOtwarty', 'DzienTygodnia', 'NazwaDnia', 'CzyNiedziela', 'CzySobota', 'CzySwieto', 'NazwaSwieta', 'Wartosc', 'Ilosc', 'LiczbaTransakcji', 'CenaDzienna', 'CenaOstatniaZnana', 'ProduktJeszczeNieSprzedawany', 'NazwaTow', 'JestWazony']

Brakujące wartości per kolumna:
TowId                                 0
Data                                  0
PierwszaSprzedaz                      0
OstatniaSprzedazTowId                 0
CzySklepOtwarty                       0
DzienTygodnia                         0
NazwaDnia                             0
CzyNiedziela                          0
CzySobota                             0
CzySwieto                             0
NazwaSwieta                     7155051
Wartosc                               0
Ilosc                                 0
LiczbaTransakcji                      0
CenaDzienna                     5799401
CenaOstatniaZ

In [7]:
pelny_kalendarz.to_parquet(
    "dane/interim/kalendarz_pelny_towid.parquet",
    compression='zstd',
    index=False
)


In [12]:
#Suma kontrolna

nazwa_pliku = "kalendarz_pelny_towid.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")

Mój hash (posortowane):   kalendarz_pelny_towid.parquet   c0928bc17428d943a4dd0573047618cae330abbf0904a01b48cbae5629c7e6d0


In [9]:
# Sprawdzenie ciągłości danych
sprawdzenie_ciaglosci = pelny_kalendarz.groupby('TowId')['Data'].agg(['min', 'max', 'count']).reset_index()
sprawdzenie_ciaglosci['OczekiwanaLiczbaDni'] = (
    (sprawdzenie_ciaglosci['max'] - sprawdzenie_ciaglosci['min']).dt.days + 1
)
sprawdzenie_ciaglosci['CzyCiagly'] = (
    sprawdzenie_ciaglosci['count'] == sprawdzenie_ciaglosci['OczekiwanaLiczbaDni']
)

print(f"TowId z ciągłym zakresem dat: {sprawdzenie_ciaglosci['CzyCiagly'].sum()}")
print(f"TowId z lukami: {(~sprawdzenie_ciaglosci['CzyCiagly']).sum()}")

print(sprawdzenie_ciaglosci[~sprawdzenie_ciaglosci['CzyCiagly']].head(10))


TowId z ciągłym zakresem dat: 12481
TowId z lukami: 0
Empty DataFrame
Columns: [TowId, min, max, count, OczekiwanaLiczbaDni, CzyCiagly]
Index: []


In [10]:
wszystkie_towid = set(df_kalendarz['TowId'].unique())
towid_ze_sprzedaza = set(df_kalendarz[df_kalendarz['TypRuchu']=='sprzedaz']['TowId'].unique())
towid_bez_sprzedazy = wszystkie_towid - towid_ze_sprzedaza

print(f"TowId bez żadnej sprzedaży: {len(towid_bez_sprzedazy):,}")

# I ile REKORDÓW (wierszy, dowolnego typu) dotyczy tych właśnie TowId
rekordy_bez_sprzedazy = df_kalendarz[df_kalendarz['TowId'].isin(towid_bez_sprzedazy)]
print(f"Rekordów (dowolnego typu) dla TowId bez sprzedaży: {len(rekordy_bez_sprzedazy):,}")


TowId bez żadnej sprzedaży: 0
Rekordów (dowolnego typu) dla TowId bez sprzedaży: 0


In [11]:
# Ile rekordów w df_kalendarz nie jest sprzedażą (np. zwroty, korekty, itp.)
print(f"Surowe dane mają {(df_kalendarz['TypRuchu'] != 'sprzedaz').sum()} rekordów, "
      f"które nie są sprzedażą (np. jedna dostawa PZ)")
print(f"Pelny kalendarz ma {(pelny_kalendarz['Wartosc']==0).sum()} rekordów z zerową wartością sprzedaży (brak sprzedaży w danym dniu)")
print(f"To {(pelny_kalendarz['Wartosc']==0).mean()*100:.1f}% wszystkich wierszy z zerową wartością sprzedaży (brak sprzedaży w danym dniu)")


Surowe dane mają 683357 rekordów, które nie są sprzedażą (np. jedna dostawa PZ)
Pelny kalendarz ma 5799401 rekordów z zerową wartością sprzedaży (brak sprzedaży w danym dniu)
To 78.1% wszystkich wierszy z zerową wartością sprzedaży (brak sprzedaży w danym dniu)
